# SETUP



In [ ]:
# Clona la repository
!git clone --branch Final https://github.com/Tomm100/Generative-Adversarial-Networks-for-Data-Augmentation-and-Domain-Adaptation.git



In [3]:
%cd Generative-Adversarial-Networks-for-Data-Augmentation-and-Domain-Adaptation

/content/Generative-Adversarial-Networks-for-Data-Augmentation-and-Domain-Adaptation


In [ ]:
!pip install -r requirements.txt

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p ./data


In [6]:
#dataset
!cp -r /content/drive/MyDrive/ProgettoMLVM/VinDr_Target_DA_Ready.zip ./data/
!cp -r /content/drive/MyDrive/ProgettoMLVM/modified_dataset.zip ./data/

#unzip
!unzip -q ./data/VinDr_Target_DA_Ready.zip -d data/vindr_target
!unzip -q ./data/modified_dataset.zip -d data/modified_dataset

#Pulizia
!rm ./data/modified_dataset.zip
!rm ./data/VinDr_Target_DA_Ready.zip

print("Estrazione completata e file temporanei rimossi!")

Estrazione completata e file temporanei rimossi!


In [ ]:
!wandb login

wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter: 
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: depaolamario898 to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


# TRAINING

ResNet-18 - Training sul dataset originale

In [ ]:
!python experiments/train_classifier.py --tag Baseline

Pipeline WGAN: Classificazione baseline + training WGAN + generazioni immagini sintetiche + classificazione su dataset augmented

In [ ]:
!python experiments/main_wgan.py

Pipeline SNGAN: Classificazione baseline + training SNGAN + generazioni immagini sintetiche + classificazione su dataset augmented

In [ ]:
!python experiments/main_sngan.py

Generazione Dataset Augmented


In [ ]:
!python dataset/build_augmented.py --gan-ckpt /content/drive/MyDrive/ProgettoMLVM/Materiale_Consegna/SNGAN_128_complete/G_epoch_210.pth --gap 75

ResNet-18 - Training sul dataset augmented

In [ ]:
!python experiments/train_classifier.py --train-dir ./results/augmented_dataset/train --tag Augmented

Ablation - % di gap colmato con immagini provenienti da classical augmentation che beneficia al classificatore (macro F1)

In [ ]:
!python experiments/main_classical_aug.py

Ablation - % di gap colmato con immagini sintetiche che beneficia al classificatore (macroF1)

In [ ]:
!python experiments/ablation_synth_study.py

DANN synth -> real

In [ ]:
!python experiments/main_dann_synth.py

DANN cross hospital

In [ ]:
!python experiments/main_dann_cross_hospital.py

# VALUTAZIONI

VALUTAZIONE MODELLO - Dato un checkpoint, valuta la macro F1



In [ ]:
!python evaluation/eval_resnet.py --ckpt /content/drive/MyDrive/ProgettoMLVM/Materiale_Consegna/Resnet_on_baseline/best_model_Phase1.pth

Valutazione dati sintetici tramite FID e KID - Dato il checkpoint di una GAN, calcola FID e KID

In [ ]:
!python evaluation/metrics_kid_fid.py --ckpt-dir [PATH CARTELLA CON TUTTI I PESI] --gan sngan128

Feature Analysys: Indaga sul domain gap tramite analisi PCA, t-SNE, UMAP

In [ ]:
!python experiments/feature_analysis.py

Curve di ROC - Confrontare le curve di ROC tra modello baseline e modello augmented

In [ ]:
# arg1 - I pesi del modello baseline
# arg2 - I pesi del modello augmented (75%)

!python evaluation/eval_roc_pr.py --baseline-ckpt /content/drive/MyDrive/ProgettoMLVM/Materiale_Consegna/Resnet_on_baseline/best_model_Phase1.pth --finale-ckpt /content/drive/MyDrive/ProgettoMLVM/Materiale_Consegna/Resnet_on_augmented/best_model_Phase3.pth

Analisi Grad-CAM - Confrontare le immagini Grad-CAM tra modello baseline e modello augmented

In [ ]:
# arg1 - I pesi del modello augmented (75%)
# arg2 - I pesi del modello baseline (per il confronto)

!python evaluation/eval_gradcam.py --ckpt /content/drive/MyDrive/ProgettoMLVM/Materiale_Consegna/Ablation_bestgan_augmentation/best_model_Ablation_75pct.pth --/content/drive/MyDrive/ProgettoMLVM/Materiale_Consegna/Resnet_on_baseline/best_model_Phase1.pth

DANN: synth -> real

In [ ]:
!python evaluation/eval_dann.py --ckpt /percorso/best_DANN_Synth.pth

DANN: cross hospital (Kermany to VinDr)

In [ ]:
!python evaluation/eval_dann_cross.py --ckpt best_DANN_lam1.pth